# LLM Warm-start × RL Engine Ablation (CSI300, seed=42)

Compares vanilla / compose / alphaagent warm-start across AlphaGen (PPO) and
AlphaQCM (qrdqn). 10 pools, two RL step counts (30k early sweep + 300k for
the alphaagent comparison).

| Condition | Engine | Warm-start | Steps | Judge filter |
|---|---|---|---|---|
| `A_qcm` | AlphaQCM | ✗ | 30k | ✗ |
| `A_qcm_300k` | AlphaQCM | ✗ | 300k | ✗ |
| `B_compose_alphagen` | AlphaGen | compose | 30k | ✗ |
| `B_compose_qcm` | AlphaQCM | compose | 30k | ✗ |
| `B_alphaagent_alphagen` | AlphaGen | alphaagent | 300k | ✗ |
| `B_alphaagent_qcm` | AlphaQCM | alphaagent | 300k | ✗ |
| `C_compose_alphagen_judge` | AlphaGen | compose | 30k | ✓ DeepSeek |
| `C_compose_qcm_judge` | AlphaQCM | compose | 30k | ✓ DeepSeek |
| `C_alphaagent_alphagen_judge` | AlphaGen | alphaagent | 300k | ✓ DeepSeek |
| `C_alphaagent_qcm_judge` | AlphaQCM | alphaagent | 300k | ✓ DeepSeek |

All LLM calls (idea-agent + judge) route through OpenRouter pinned to the
**official DeepSeek** provider, model `deepseek/deepseek-v4-pro`. The
alphaagent regulator is a faithful port of FactorRegulator from
[AlphaAgent KDD 2025](https://github.com/RndmVariableQ/AlphaAgent):
structural AST novelty (largest common subtree ≤ 8 nodes vs the 80-factor
library) plus free-args / unique-vars leaf-ratio gates.


In [ ]:
from __future__ import annotations
import json, sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "external" / "alphagen"))

# ──────────────────────────────────────────────────────────────
# CONFIG — 10 conditions × 1 seed
# ──────────────────────────────────────────────────────────────
DATA_SOURCE = "cn"
SEEDS = [42]

ENGINE = {
    # 30k sweep
    "A_qcm":                          "qcm",
    "B_compose_alphagen":             "alphagen",
    "B_compose_qcm":                  "qcm",
    "C_compose_alphagen_judge":       "alphagen",
    "C_compose_qcm_judge":            "qcm",
    # 300k alphaagent comparison
    "A_qcm_300k":                     "qcm",
    "B_alphaagent_alphagen":          "alphagen",
    "B_alphaagent_qcm":               "qcm",
    "C_alphaagent_alphagen_judge":    "alphagen",
    "C_alphaagent_qcm_judge":         "qcm",
}
# Display step counts so the table is unambiguous
STEPS = {c: 300_000 if ("300k" in c or "alphaagent" in c) else 30_000
         for c in ENGINE}
CONDITIONS = list(ENGINE.keys())

FACT_DIR = ROOT / "data" / "factors"
TB_ALPHAGEN_DIR = ROOT / "out" / "tensorboard"
TB_QCM_DIR = ROOT / "data" / "qcm_logs" / "pool_20_QCM_1.0"

from alphagen.config import OPERATORS
from alphagen.data.parser import ExpressionParser
from src.factor_mining._calc_factory import build_calculators


def pool_path(cond, seed):
    """Resolve condition name → actual pool JSON path.

    Judge-filtered pools are saved as <prefix>_judge_pool.json (suffix-after-seed);
    everything else is <name>_cn_seed{S}_pool.json.
    """
    if cond.endswith("_judge"):
        # e.g. C_compose_alphagen_judge → C_compose_alphagen_cn_seed42_judge_pool.json
        stem = cond[:-len("_judge")]
        return FACT_DIR / f"{stem}_cn_seed{seed}_judge_pool.json"
    return FACT_DIR / f"{cond}_cn_seed{seed}_pool.json"


def load_pool(cond, seed):
    p = pool_path(cond, seed)
    return json.loads(p.read_text()) if p.exists() else None


pools = {(c, s): load_pool(c, s) for c in CONDITIONS for s in SEEDS}
for k, v in pools.items():
    status = "OK" if v else "MISSING"
    print(f"{k}: {status:<7}  ({pool_path(*k).name})")


## 1. Headline summary table

In [ ]:
calcs = build_calculators(
    data_source=DATA_SOURCE,
    data_config_path="../config/data_config.yaml",
    splits_to_load=("train", "test"),
)
train_calc = calcs["train"]
test_calc = calcs["test"]
parser = ExpressionParser(
    operators=OPERATORS,
    ignore_case=False,
    time_deltas_need_suffix=False,
    non_positive_time_deltas_allowed=False,
    feature_need_dollar_sign=True,
)

def pool_metrics(pool: dict) -> dict:
    parsed = []
    for e in pool["exprs"]:
        try:
            parsed.append(parser.parse(e))
        except Exception:
            continue
    if not parsed:
        return {"pool_size": 0}
    single_ics_test  = [float(test_calc.calc_single_IC_ret(e))  for e in parsed]
    single_ics_train = [float(train_calc.calc_single_IC_ret(e)) for e in parsed]
    abs_test = sorted([abs(x) for x in single_ics_test], reverse=True)
    if len(parsed) < 2:
        diversity = float("nan")
    else:
        mat = np.stack([train_calc.evaluate_alpha(e).cpu().numpy().ravel() for e in parsed])
        mat = np.nan_to_num(mat, nan=0.0)
        corr = np.corrcoef(mat)
        iu = np.triu_indices_from(corr, k=1)
        diversity = float(np.nanmean(np.abs(corr[iu])))
    return {
        "pool_size": len(parsed),
        "val_ic":  pool.get("val_ic"),
        "val_ric": pool.get("val_ric"),
        "test_ic": pool.get("test_ic"),
        "test_ric": pool.get("test_ric"),
        "top5_test_abs_single_ic": float(np.mean(abs_test[:5])) if abs_test else float("nan"),
        "mean_train_abs_single_ic": float(np.mean(np.abs(single_ics_train))) if single_ics_train else float("nan"),
        "pool_mean_abs_corr": diversity,
    }

rows = []
for (c, s), p in pools.items():
    if p is None:
        rows.append({"condition": c, "seed": s, "engine": ENGINE[c],
                     "steps": STEPS[c], "status": "MISSING"})
        continue
    rows.append({"condition": c, "seed": s, "engine": ENGINE[c],
                 "steps": STEPS[c], "status": "OK", **pool_metrics(p)})
raw = pd.DataFrame(rows)
raw


In [ ]:
# Tidy per-condition view sorted by warm-start mode + engine
order_cols = ["engine", "steps", "status", "pool_size",
              "val_ic", "test_ic", "test_ric",
              "top5_test_abs_single_ic", "pool_mean_abs_corr"]
view = raw.set_index("condition")[order_cols].reindex(CONDITIONS)
view.round(4)


## 2. Learning curves — RL training only

AlphaGen → `out/tensorboard/<run>_1/` tag `test/ic_mean`.
AlphaQCM → `data/qcm_logs/pool_20_QCM_1.0/<run>/summary/` tag `ic/test`.
C_*_judge pools are post-hoc filters, no training curve.


In [ ]:
try:
    from tbparse import SummaryReader
    HAVE_TB = True
except ImportError:
    print('tbparse not installed. `pip install tbparse` for learning curves.')
    HAVE_TB = False

# Only conditions that actually trained an RL agent
TRAINING_CONDITIONS = [c for c in CONDITIONS if not c.startswith("C_")]

def load_alphagen_curve(cond: str, seed: int) -> pd.DataFrame:
    runs = sorted(TB_ALPHAGEN_DIR.glob(f"{cond}_cn_seed{seed}_*"))
    if not runs:
        return pd.DataFrame()
    df = SummaryReader(str(runs[-1]), pivot=False).scalars
    return df[df["tag"] == "test/ic_mean"][["step", "value"]].copy()

def load_qcm_curve(cond: str, seed: int) -> pd.DataFrame:
    run = TB_QCM_DIR / f"{cond}_cn_seed{seed}" / "summary"
    if not run.exists():
        return pd.DataFrame()
    df = SummaryReader(str(run), pivot=False).scalars
    return df[df["tag"] == "ic/test"][["step", "value"]].copy()

curves = {}
if HAVE_TB:
    for cond in TRAINING_CONDITIONS:
        loader = load_alphagen_curve if ENGINE[cond] == "alphagen" else load_qcm_curve
        for s in SEEDS:
            curves[(cond, s)] = loader(cond, s)
    for k, v in curves.items():
        print(f"{k}: {len(v):>5} points, engine={ENGINE[k[0]]}, steps={STEPS[k[0]]}")


In [ ]:
if HAVE_TB and any(len(df) for df in curves.values()):
    palette = {
        "A_qcm":                 "tab:gray",
        "A_qcm_300k":            "black",
        "B_compose_alphagen":    "tab:blue",
        "B_compose_qcm":         "tab:cyan",
        "B_alphaagent_alphagen": "tab:red",
        "B_alphaagent_qcm":      "tab:orange",
    }
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for cond in TRAINING_CONDITIONS:
        series = []
        for s in SEEDS:
            df = curves.get((cond, s), pd.DataFrame())
            if df.empty:
                continue
            series.append(pd.Series(df["value"].values, index=df["step"].values))
        if not series:
            continue
        aligned = pd.concat(series, axis=1).ffill()
        mean = aligned.mean(axis=1)
        ax.plot(mean.index, mean.values,
                color=palette.get(cond, "k"),
                label=f"{cond} ({ENGINE[cond]}, {STEPS[cond]//1000}k)",
                linewidth=1.8)
    ax.set_xlabel("Training step")
    ax.set_ylabel("Test IC (ensemble or pool mean)")
    ax.set_title("CSI300 — warm-start mode × RL engine (seed 42)")
    ax.grid(alpha=0.3)
    ax.axhline(0, color="k", linewidth=0.5, alpha=0.4)
    ax.legend(loc="best", fontsize=8, ncol=2)
    plt.tight_layout()
    out_png = ROOT / "out" / "ablation_learning_curves.png"
    out_png.parent.mkdir(parents=True, exist_ok=True)
    plt.savefig(out_png, dpi=120)
    print(f"Saved {out_png}")
    plt.show()
else:
    print("No tensorboard data to plot.")


## 3. Judge-dropped factors

DeepSeek scored each expression in the B_compose_* pools; expressions below
the threshold (kept_top_k excepted) were dropped from the C_compose_*_judge
pools. List what got cut and why.


In [ ]:
JUDGE_CONDS = ["C_compose_alphagen_judge", "C_compose_qcm_judge",
               "C_alphaagent_alphagen_judge", "C_alphaagent_qcm_judge"]

for c_cond in JUDGE_CONDS:
    for s in SEEDS:
        p = pools.get((c_cond, s))
        if p is None or "filter" not in p:
            print(f"[{c_cond} seed={s}] no filter metadata")
            continue
        info = p["filter"]
        print(f"\n[{c_cond} seed={s}]  kept={info['n_kept']}/{info['n_input']}  "
              f"threshold={info['threshold']}")
        print("-- dropped --")
        for r in info.get("judge_results", []):
            if not r["kept"]:
                print(f"  score={r['score']:.2f}  {r['expr'][:120]}")
                if r.get("nl"):
                    print(f"             NL: {r['nl'][:120]}")


## 4. Warm-seed quality (idea-agent / DeepSeek compose output)


In [ ]:
WARM_FILES = [
    ("compose",    "warm_seeds_cn_compose_deepseek_seed{}.json"),
    ("compose",    "warm_seeds_cn_compose_seed{}.json"),  # fallback
    ("alphaagent", "warm_seeds_cn_alphaagent_deepseek_seed{}.json"),
]

for label, pattern in WARM_FILES:
    for s in SEEDS:
        p = FACT_DIR / pattern.format(s)
        if not p.exists():
            continue
        payload = json.loads(p.read_text())
        print(f"\n=== {label}  seed={s}  file={p.name}")
        print(f"  mode={payload.get('mode')} backend={payload.get('llm_backend')} "
              f"model={payload.get('model')}")
        print(f"  selected={payload.get('n_selected','?')} / "
              f"resolved={payload.get('n_resolved','?')} / "
              f"library={payload.get('n_library','?')}")
        for sd in payload.get("seeds", []):
            fam = sd.get("family", sd.get("hypothesis", ""))
            print(f"  IC={sd['train_ic']:+.4f}  [{fam:>14s}]  {sd['expr']}")
        break  # only show first matching path per label


## 5. Final write-up table


In [ ]:
def _fmt(v):
    return "  n/a " if v is None or (isinstance(v, float) and pd.isna(v)) else f"{float(v):+.4f}"

summary_rows = []
for c in CONDITIONS:
    row = raw[raw.condition == c].head(1)
    if row.empty or row.iloc[0].get("status") != "OK":
        summary_rows.append({"condition": c, "engine": ENGINE[c],
                             "steps": f"{STEPS[c]//1000}k",
                             "size": "—", "test_IC": "MISSING",
                             "val_IC": "—", "top5_|IC|": "—", "mean_|corr|": "—"})
        continue
    r = row.iloc[0]
    summary_rows.append({
        "condition":     c,
        "engine":        ENGINE[c],
        "steps":         f"{STEPS[c]//1000}k",
        "size":          int(r["pool_size"]),
        "test_IC":       _fmt(r["test_ic"]),
        "val_IC":        _fmt(r["val_ic"]),
        "top5_|IC|":     _fmt(r["top5_test_abs_single_ic"]),
        "mean_|corr|":   _fmt(r["pool_mean_abs_corr"]),
    })
summary = pd.DataFrame(summary_rows).set_index("condition")
summary
